# Stage 10: fine-tuning v2 (Colab, A100-40GB)

Runs the GPU steps of `reports/Long context extraction fine tuning.md`, in this order:

| Part | Runtime | Action plan step |
|---|---|---|
| A | inference (vLLM) | 2: BASE vs Stage 7 adapter on **validation**, identical settings, adapter loaded correctly |
| B | training | 6: preflight, memory probe, LoRA v2 on Qwen3-8B (primary) and Mistral-7B (replication) |
| C | inference (vLLM) | 7: base and per-epoch adapters on validation, constrained chunked extraction, checkpoint selection |
| D | CPU | 8: downstream evaluation on validation |
| E | inference | 9: test split, **once per pre-registered arm** (`extract_v2.py` refuses a second run) |

Gold v1.4.1: 42 studies, 246 rows; split 30/6/6 studies. 29 train studies have source text (the Tang 2025 study has none), so the training set is built from 29 studies, never from all 42.

vLLM and the training stack pin different torch versions: run A, C and E in one runtime and B in another (Runtime > Restart between them).

In [ ]:
# Setup (every runtime)
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!git clone -b feat/finetuning-v2-pipeline https://github.com/dedemerve/ILSA-LLM-Extractor.git repo || (cd repo && git pull)
%cd /content/repo
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
STAGE7_ADAPTER = '/content/drive/MyDrive/ILSA/lora_adapter_final'   # adjust to where the Stage 7 adapter lives

## A. Inference runtime: step 2 (re-run BASE and the Stage 7 adapter on validation)

In [ ]:
!pip install -q vllm==0.31.0 transformers==5.17.0 openai==3.26.1 pydantic scikit-learn pandas scipy
!python3 scripts/finetuning/preflight_checks.py --adapter $STAGE7_ADAPTER || echo 'expected: Stage 9 loading path loads 0 tensors'

In [ ]:
%%bash -s "$STAGE7_ADAPTER"
set -e
M=mistralai/Mistral-7B-Instruct-v0.3
python3 scripts/finetuning/extract_v2.py --mode document --backend vllm --model $M --split val --arm base_mistral_doc
python3 scripts/finetuning/extract_v2.py --mode document --backend vllm --model $M --adapter $1 --split val --arm stage7_lora_doc
python3 scripts/finetuning/eval_extraction.py --split val \
  --pred outputs/finetuning/v2/predictions/base_mistral_doc_val.json --label base \
  --pred-b outputs/finetuning/v2/predictions/stage7_lora_doc_val.json --label-b stage7_lora \
  --out outputs/finetuning/v2/eval/step2_base_vs_stage7_val.json > /dev/null
python3 -c "import json;d=json.load(open('outputs/finetuning/v2/eval/step2_base_vs_stage7_val.json'));[print(k, d[k]['summary']['strict'], d[k]['summary']['lenient']) for k in ('base','stage7_lora')];print(d['delta_stage7_lora_minus_base'])"

## B. Training runtime: step 6

In [ ]:
!pip install -q -r requirements-finetuning-v2.txt
# The SFT data is committed (outputs/finetuning/v2/data/sft_*.jsonl). Do not rebuild it here:
# build_sft_v2.py needs the Stage 7 JSONL files, which are git-ignored and exist only locally.
!python3 scripts/finetuning/preflight_checks.py --data outputs/finetuning/v2/data/sft_train.jsonl --tokenizer Qwen/Qwen3-8B --max-length 8192
!python3 scripts/finetuning/preflight_checks.py --data outputs/finetuning/v2/data/sft_train.jsonl --tokenizer mistralai/Mistral-7B-Instruct-v0.3 --pad-token unk --max-length 8192

In [ ]:
# memory probe: 2 steps on the longest example
!python3 scripts/finetuning/train_lora_v2.py --model Qwen/Qwen3-8B --memory-probe --out /content/probe

In [ ]:
%%bash
set -e
# primary arm and replication arm; grid kept small on purpose (N = 29 studies)
for LR in 1e-4 2e-4; do
  python3 scripts/finetuning/train_lora_v2.py --model Qwen/Qwen3-8B --lr $LR --epochs 3 \
    --out outputs/finetuning/v2/runs/qwen3_8b_r16_lr$LR
done
python3 scripts/finetuning/train_lora_v2.py --model mistralai/Mistral-7B-Instruct-v0.3 --lr 1e-4 --epochs 3 \
  --out outputs/finetuning/v2/runs/mistral7b_r16_lr1e-4
cp -r outputs/finetuning/v2/runs /content/drive/MyDrive/ILSA/stage10_runs

## C. Inference runtime: step 7 (validation only) and checkpoint selection

In [ ]:
%%bash
set -e
pip install -q vllm==0.31.0 transformers==5.17.0 openai==3.26.1 pydantic scikit-learn pandas scipy
cp -r /content/drive/MyDrive/ILSA/stage10_runs outputs/finetuning/v2/runs
python3 scripts/finetuning/extract_v2.py --model Qwen/Qwen3-8B --split val --arm base_qwen3_chunked
python3 scripts/finetuning/extract_v2.py --model mistralai/Mistral-7B-Instruct-v0.3 --split val --arm base_mistral_chunked
for RUN in outputs/finetuning/v2/runs/*/; do
  MODEL=$(python3 -c "import json;print(json.load(open('$RUN/run_manifest.json'))['model'])")
  for CK in $RUN/checkpoint-*; do
    ARM=$(basename $RUN)_$(basename $CK)
    python3 scripts/finetuning/extract_v2.py --model $MODEL --adapter $CK --split val --arm $ARM
  done
done
python3 scripts/finetuning/select_checkpoint.py --split val

## D. Downstream evaluation on validation (CPU is enough)

In [ ]:
!python3 scripts/finetuning/downstream_eval.py --split val \
   --condition base=outputs/finetuning/v2/predictions/base_qwen3_chunked_val.json \
   --condition lora=$(python3 -c "import json;print(json.load(open('outputs/finetuning/v2/eval/checkpoint_selection_val.json'))['selected']['predictions'])") \
   --out outputs/finetuning/v2/downstream/val.json
!python3 scripts/finetuning/downstream_eval.py --split all --mode standalone --noise --reps 100 \
   --out outputs/finetuning/v2/downstream/noise_all_standalone.json

## E. Test split: once per arm

Only after the protocol (`outputs/finetuning/EVALUATION_PROTOCOL_v2.md`) is committed and the arms below are fixed.
`--final-test-run` records the arm in `outputs/finetuning/v2/test_runs.json`; a second run of the same arm is refused.

In [ ]:
# %%bash
# python3 scripts/finetuning/extract_v2.py --model Qwen/Qwen3-8B --split test --arm base_qwen3_chunked --final-test-run
# python3 scripts/finetuning/extract_v2.py --model Qwen/Qwen3-8B --adapter <selected checkpoint> --split test --arm qwen3_lora_selected --final-test-run